In [5]:
from google.colab import drive

drive.mount('/content/drive')


KeyboardInterrupt: 

In [40]:
from pathlib import Path

REPO_URL = 'https://github.com/CameronBadman/Hippo-encoder.git'
WORKDIR = Path('/content/Hippo-encoder')

if not WORKDIR.exists():
    !git clone {REPO_URL} {WORKDIR}

%cd /content/Hippo-encoder
!git pull --ff-only || true


/content/Hippo-encoder
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 7 (delta 2), reused 7 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 3.08 KiB | 1.03 MiB/s, done.
From https://github.com/CameronBadman/Hippo-encoder
   a2a92d6..fe48a0e  main       -> origin/main
Updating a2a92d6..fe48a0e
Fast-forward
 README.md                              |  12 ++
 benchmarks/sample_region_cases.json    |  50 +++++++++
 scripts/benchmark_region_membership.py | 193 +++++++++++++++++++++++++++++++++
 3 files changed, 255 insertions(+)
 create mode 100644 benchmarks/sample_region_cases.json
 create mode 100644 scripts/benchmark_region_membership.py


In [23]:
%cd /content/Hippo-encoder
!python -m pip install --upgrade pip
!pip install -e .


/content/Hippo-encoder
Obtaining file:///content/Hippo-encoder
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hippo-encoder (pyproject.toml) ... done
  Created wheel for hippo-encoder: filename=hippo_encoder-0.1.0-0.editable-py3-none-any.whl size=2413 sha256=a50e76bb0351206d39924743dcd88da1c623708d6324641ec8171031a538f909
  Stored in directory: /tmp/pip-ephem-wheel-cache-xh0kq0ld/wheels/97/19/24/8366964dd5be8ff3aa53cd6100f1344ee01af98a718baaae20
Successfully built hippo-encoder
  Attempting uninstall: hippo-encoder
    Found existing installation: hippo-encoder 0.1.0
    Uninstalling hippo-encoder-0.1.0:
      Successfully uninstalled hippo-encoder-0.1.0


In [24]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/hippo_encoder_data')
IMAGE_ROOT = DATA_ROOT / 'images'
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
OUTPUT_DIR = Path('/content/drive/MyDrive/hippo_encoder_runs/distill-clip-tiny')

IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATA_ROOT)
print('Image root:', IMAGE_ROOT)
print('Train manifest:', TRAIN_JSONL)
print('Output dir:', OUTPUT_DIR)


Dataset root: /content/drive/MyDrive/hippo_encoder_data
Image root: /content/drive/MyDrive/hippo_encoder_data/images
Train manifest: /content/drive/MyDrive/hippo_encoder_data/train.jsonl
Output dir: /content/drive/MyDrive/hippo_encoder_runs/distill-clip-tiny


In [25]:
from pathlib import Path

TRAIN_JSONL = Path('/content/drive/MyDrive/hippo_encoder_data/train.jsonl')

if not TRAIN_JSONL.exists():
    TRAIN_JSONL.parent.mkdir(parents=True, exist_ok=True)
    TRAIN_JSONL.write_text(
        '{"image":"example.jpg","text":"a red chair near the window"}\n',
        encoding='utf-8',
    )
    print('Wrote sample manifest to', TRAIN_JSONL)
else:
    print('Manifest already exists:', TRAIN_JSONL)


Manifest already exists: /content/drive/MyDrive/hippo_encoder_data/train.jsonl


In [36]:
from pathlib import Path
import json

config = {
    "teacher_model_name": "intfloat/e5-base-v2",
    "student_model_name": "BAAI/bge-small-en-v1.5",
    "dataset_jsonl": "/content/drive/MyDrive/hippo_encoder_data/train.jsonl",
    "output_dir": "/content/drive/MyDrive/hippo_encoder_runs/distill-text-regression-only",
    "max_text_length": 64,
    "batch_size": 32,
    "num_epochs": 3,
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "log_every": 20,
    "save_every": 100000,
    "num_workers": 2,
    "teacher_text_weight": 1.0,
    "hidden_state_weight": 0.2,
    "contrastive_weight": 0.0,
    "normalize_targets": True,
    "gradient_clip_norm": 1.0,
    "warmup_steps": 100,
    "seed": 42
  }

runtime_config_path = Path('/content/Hippo-encoder/configs/colab_run.json')
runtime_config_path.parent.mkdir(parents=True, exist_ok=True)
runtime_config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(runtime_config_path.read_text())


{
  "teacher_model_name": "intfloat/e5-base-v2",
  "student_model_name": "distilgpt2",
  "dataset_jsonl": "/content/drive/MyDrive/hippo_encoder_data/train.jsonl",
  "output_dir": "/content/drive/MyDrive/hippo_encoder_runs/distill-text-regression-only",
  "max_text_length": 64,
  "batch_size": 32,
  "num_epochs": 3,
  "learning_rate": 0.0001,
  "weight_decay": 0.01,
  "log_every": 20,
  "save_every": 100000,
  "num_workers": 2,
  "teacher_text_weight": 1.0,
  "hidden_state_weight": 0.2,
  "contrastive_weight": 0.0,
  "normalize_targets": true,
  "gradient_clip_norm": 1.0,
  "warmup_steps": 100,
  "seed": 42
}


In [34]:
%cd /content/Hippo-encoder
!python scripts/prepare_text_dataset.py \
--dataset ag_news \
--split train \
--text-column text \
--limit 20000 \
--prefix 'passage: ' \
--output /content/drive/MyDrive/hippo_encoder_data/train.jsonl

/content/Hippo-encoder
{
  "dataset": "ag_news",
  "split": "train",
  "text_column": "text",
  "rows_written": 20000,
  "output": "/content/drive/MyDrive/hippo_encoder_data/train.jsonl"
}


In [35]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config configs/colab_run.json


/content/Hippo-encoder
Loading weights: 100% 199/199 [00:00<00:00, 1024.11it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 76/76 [00:00<00:00, 5768.29it/s, Materializing param=wte.weight]
GPT2Model LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
epoch=0 step=20 loss=1.1426 text=0.9447 hidden=0.9898 contrastive=3.4657
epoch=0 step=40 loss=0.5843 text=0.4575 hidden=0.6342 contrastive=3.4647
ep

In [38]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

teacher_name = "intfloat/e5-base-v2"
student_ckpt = "/content/drive/MyDrive/hippo_encoder_runs/distill-text-regression-only/epoch-2"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)
teacher_model = AutoModel.from_pretrained(teacher_name).to(device).eval()

student_tokenizer = AutoTokenizer.from_pretrained(f"{student_ckpt}/tokenizer")
student_backbone = AutoModel.from_pretrained(f"{student_ckpt}/backbone").to(device).eval()

heads = torch.load(f"{student_ckpt}/heads.pt", map_location=device)
embed_head = torch.nn.Linear(
    student_backbone.config.hidden_size,
    teacher_model.config.hidden_size,
).to(device)
embed_head.load_state_dict(heads["embed_head"])
embed_head.eval()

def masked_mean(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    masked = hidden_states * mask
    denom = mask.sum(dim=1).clamp(min=1.0)
    return masked.sum(dim=1) / denom

@torch.no_grad()
def teacher_embed(texts):
    batch = teacher_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    out = teacher_model(**batch, return_dict=True)
    emb = masked_mean(out.last_hidden_state, batch["attention_mask"])
    return F.normalize(emb, dim=-1)

@torch.no_grad()
def student_embed(texts):
    batch = student_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    out = student_backbone(**batch, return_dict=True)
    pooled = masked_mean(out.last_hidden_state, batch["attention_mask"])
    emb = embed_head(pooled)
    return F.normalize(emb, dim=-1)

eval_texts = [
    "query: red chair near the window",
    "query: a scarlet chair beside the window",
    "query: blue mug on the desk",
    "query: coffee cup sitting on the office desk",
    "query: dog running through the grass",
    "query: a small dog sprinting across a field",
    "query: stock market falls after weak earnings",
    "query: company shares drop after poor results",
    "query: basketball team wins championship game",
    "query: the team secured the title in the final match",
    "query: person standing beside a parked bicycle",
    "query: someone next to a bike on the street",
]

t = teacher_embed(eval_texts)
s = student_embed(eval_texts)

pair_cos = F.cosine_similarity(t, s, dim=-1)
print("\nTeacher vs student cosine per item:")
for text, score in zip(eval_texts, pair_cos.tolist()):
    print(f"{score:.4f}  {text}")

print("\nMean teacher-student cosine:", pair_cos.mean().item())

teacher_sim = t @ t.T
student_sim = s @ s.T

def topk_neighbors(sim, k=3):
    idx = torch.topk(sim, k=k+1, dim=-1).indices
    return idx[:, 1:]

teacher_nn = topk_neighbors(teacher_sim, k=3)
student_nn = topk_neighbors(student_sim, k=3)

print("\nTop-3 neighbor overlap:")
overlaps = []
for i, text in enumerate(eval_texts):
    a = set(teacher_nn[i].tolist())
    b = set(student_nn[i].tolist())
    overlap = len(a & b) / len(a)
    overlaps.append(overlap)
    print(f"{overlap:.2f}  {text}")
    print("  teacher:", [eval_texts[j] for j in teacher_nn[i].tolist()])
    print("  student:", [eval_texts[j] for j in student_nn[i].tolist()])

print("\nMean top-3 overlap:", sum(overlaps) / len(overlaps))

pairs = [
    (0, 1, "chair paraphrase"),
    (2, 3, "mug paraphrase"),
    (4, 5, "dog paraphrase"),
    (6, 7, "market paraphrase"),
    (8, 9, "basketball paraphrase"),
    (10, 11, "bicycle paraphrase"),
]

print("\nParaphrase similarity check:")
teacher_pair_scores = []
student_pair_scores = []
for i, j, label in pairs:
    ts = F.cosine_similarity(t[i:i+1], t[j:j+1]).item()
    ss = F.cosine_similarity(s[i:i+1], s[j:j+1]).item()
    teacher_pair_scores.append(ts)
    student_pair_scores.append(ss)
    print(f"{label:22s} teacher={ts:.4f} student={ss:.4f}")

print("\nMean paraphrase cosine:")
print("teacher:", sum(teacher_pair_scores) / len(teacher_pair_scores))
print("student:", sum(student_pair_scores) / len(student_pair_scores))

device: cuda
gpu: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


Teacher vs student cosine per item:
0.7580  query: red chair near the window
0.7634  query: a scarlet chair beside the window
0.7624  query: blue mug on the desk
0.7640  query: coffee cup sitting on the office desk
0.7314  query: dog running through the grass
0.7356  query: a small dog sprinting across a field
0.8182  query: stock market falls after weak earnings
0.8661  query: company shares drop after poor results
0.7679  query: basketball team wins championship game
0.8302  query: the team secured the title in the final match
0.7643  query: person standing beside a parked bicycle
0.7800  query: someone next to a bike on the street

Mean teacher-student cosine: 0.7784520983695984

Top-3 neighbor overlap:
0.67  query: red chair near the window
  teacher: ['query: a scarlet chair beside the window', 'query: coffee cup sitting on the office desk', 'query: blue mug on the desk']
  student: ['query: a scarlet chair beside the window', 'query: blue mug on the desk', 'query: person standin

In [42]:
%cd /content/Hippo-encoder
!git pull --ff-only
!python scripts/benchmark_region_membership.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint /content/drive/MyDrive/hippo_encoder_runs/distill-text-regression-only/epoch-2

/content/Hippo-encoder
Already up to date.
Loading weights: 100% 199/199 [00:00<00:00, 1043.40it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 76/76 [00:00<00:00, 2039.84it/s, Materializing param=wte.weight]
{
  "summary": {
    "student_teacher_cosine": 0.7674965411424637,
    "teacher_positive_hit_rate": 1.0,
    "teacher_negative_false_positive_rate": 0.0,
    "student_positive_hit_rate": 0.0,
    "student_negative_false_positive_rate": 0.0,
    "teacher_positive_inside_fraction_mean": 1.0,
    "teacher_negative_inside_fraction_mean": 0.4648437723517418,
    "student_positive_inside_fraction_mean": 0.5263672024011612,
    "student_negative_inside_fraction_mean": 0.503472

## Notes

- Replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub account.
- Put your real images in `/content/drive/MyDrive/hippo_encoder_data/images`.
- Update the JSONL manifest so each row has `image` and `text`.
- If you switch to a different student under 1B params, only the config needs to change.